# Data Cleaning Report — Tankerkönig Fuel Prices (December 2025)

This notebook cleans and aggregates raw per-minute fuel price data from the Tankerkönig API for ~15,000 German filling stations over December 2025, and prepares three datasets used in the Tableau report:

| Output file | Grain | Used for |
|---|---|---|
| `tankkoenig_hourly_prices.csv` | station × hour | daily price-pattern charts |
| `tankkoenig_daily_prices.csv` | station × day | monthly trend chart |
| `tankkoenig_station_times_prices.csv` | station (full-month avg) | regional / brand comparison |

**Pipeline:** load → drop unused columns → fix timestamps → remove invalid prices → check duplicates → aggregate to hourly/daily/station level → export.

*Raw data isn't redistributed in this repo, per the Tankerkönig API license — see the project README.*

In [1]:
import numpy as np # Linear algebra
import pandas as pd # for working with databases
import os # for reading multiple files

## 1. Loading the data

The raw export consists of **31 daily CSV files** (one per day of December 2025), each holding per-station price snapshots for that day, pulled from the Tankerkönig API.

In [2]:
path = '/kaggle/input/datasets/vladyslavyakymchuk/tankkoenig-december'
files = os.listdir(path)
print(len(files), "files found")

31 files found


## 2. Combining into one DataFrame

All 31 daily files are concatenated into a single DataFrame — one row per price update per station.

In [3]:
dfs = []

for file in files:
    if file.endswith(".csv"):
        df_temp = pd.read_csv(os.path.join(path, file))
        dfs.append(df_temp)

df = pd.concat(dfs, ignore_index=True)

print("Size of splited DataFrame:", df.shape)
df.head()


Size of splited DataFrame: (13648295, 8)


,date,station_uuid,diesel,e5,e10,dieselchange,e5change,e10change
0,2025-12-10 00:00:24+01,3116ea83-358d-4528-a440-6a84f56cde37,1.549,1.689,1.629,1,1,1
1,2025-12-10 00:01:25+01,00060166-0002-4444-8888-acdc00000002,1.624,1.694,1.634,1,1,1
2,2025-12-10 00:01:25+01,00060728-0003-4444-8888-acdc00000003,1.624,1.694,1.634,1,1,1
3,2025-12-10 00:01:25+01,89a464bc-b992-4eb5-8888-aa6a4cf51d98,1.609,1.679,1.619,1,1,1
4,2025-12-10 00:01:25+01,7d9462a3-03f6-4363-ae1c-08341249b929,1.534,1.674,1.614,1,1,1


**Result:** 13,648,295 rows × 8 columns loaded.

## 3. Cleaning

Before aggregating, the raw data needs three fixes: dropping columns that aren't used in this analysis, normalizing timestamps to a single timezone, and removing physically-impossible prices.

**Problem:** `dieselchange`, `e5change`, `e10change` are boolean flags from the source API marking *that* a price changed since the previous update — not useful here, since the actual price values are what's being analyzed.

**Action:** dropped all three columns.

**Problem:** timestamps arrive as strings with a bare UTC offset (e.g. `+01`), which isn't a real timezone name and can't be safely compared or grouped by local hour.

**Action:** parsed to UTC, converted to `Europe/Berlin` (so hourly patterns reflect local time, including any DST shift), then stripped of timezone info for simpler downstream grouping.

In [4]:
df = df.drop(columns=['dieselchange', 'e5change', 'e10change'])

In [5]:
#Date normalazing

df['date'] = pd.to_datetime(df['date'], errors='coerce', utc=True)  # convert everything to UTC
df['date'] = df['date'].dt.tz_convert('Europe/Berlin')  # convert at the right time
df['date'] = df['date'].dt.tz_localize(None)  # remove UTC info

**Check:** did any timestamp fail to parse during the UTC → Europe/Berlin conversion above?

In [6]:
df['date'].isna().sum()

0

**Result:** 0 unparseable timestamps — all 13,648,295 rows kept.

## 4. Removing invalid prices

First, look at the distributions to spot anything that isn't physically possible.

In [7]:
df.describe()

,date,diesel,e5,e10
count,13648295,1.364830e+07,1.364830e+07,1.364830e+07
mean,2025-12-16 09:52:06.058587648,1.586595e+00,1.671054e+00,1.570400e+00
min,2025-12-01 00:00:32,-1.000000e-03,-1.000000e-03,-1.000000e-03
25%,2025-12-08 15:37:58,1.539000e+00,1.659000e+00,1.599000e+00
50%,2025-12-16 12:22:47,1.579000e+00,1.689000e+00,1.629000e+00
75%,2025-12-23 17:42:53,1.619000e+00,1.729000e+00,1.669000e+00
max,2025-12-31 23:59:07,3.330000e+00,4.444000e+00,3.333000e+00
std,NaN,7.100377e-02,2.213904e-01,3.356668e-01


**Finding:** the minimum value above is **-0.001 €** for diesel, E5 and E10 — a negative fuel price isn't real; this looks like a placeholder/sentinel value from the source feed rather than an actual price.

**Action:** replace any price ≤ 0 with `NaN`, so these rows keep their other valid columns (station, timestamp) and are automatically excluded from averages instead of distorting them.

In [8]:
df['diesel'] = df['diesel'].mask(df['diesel'] <= 0, np.nan)
df['e5'] = df['e5'].mask(df['e5'] <= 0, np.nan)
df['e10'] = df['e10'].mask(df['e10'] <= 0, np.nan)

In [9]:
df.describe()

,date,diesel,e5,e10
count,13648295,1.364458e+07,1.342897e+07,1.306835e+07
mean,2025-12-16 09:52:06.058587648,1.587027e+00,1.698345e+00,1.640090e+00
min,2025-12-01 00:00:32,1.179000e+00,1.109000e+00,1.190000e+00
25%,2025-12-08 15:37:58,1.539000e+00,1.659000e+00,1.599000e+00
50%,2025-12-16 12:22:47,1.579000e+00,1.689000e+00,1.629000e+00
75%,2025-12-23 17:42:53,1.619000e+00,1.729000e+00,1.669000e+00
max,2025-12-31 23:59:07,3.330000e+00,4.444000e+00,3.333000e+00
std,NaN,6.601294e-02,5.885015e-02,5.808169e-02


**Result:** after masking, minimum prices are now realistic (diesel ≥ 1.18€, E5 ≥ 1.11€, E10 ≥ 1.19€). The count of masked values is checked next.

**Check:** how many prices were masked to `NaN` in the step above?

In [10]:
df['diesel'].isna().sum()

3713

In [11]:
df['e5'].isna().sum()

219323

In [12]:
df['e10'].isna().sum()

579942

**Result:** invalid/missing prices affect diesel far less than E10:

| Fuel | Missing after cleaning | % of 13.65M rows |
|---|---|---|
| Diesel | 3,713 | 0.03% |
| E5 | 219,323 | 1.6% |
| E10 | 579,942 | 4.2% |

**Interpretation:** this is unlikely to be a data-quality problem. E10 is a newer fuel grade that not every station sells, so a missing E10 value most likely means *this station doesn't offer E10*, not *invalid entry*. Diesel is sold almost everywhere, which matches its near-zero missing rate.

Let's also check for duplicate readings (same station, same timestamp):

In [13]:
duplicates = df.duplicated(subset=['station_uuid', 'date']).sum()
print(f"Number of dublicates: {duplicates}")

Number of dublicates: 0


**Result:** 0 duplicate (station, timestamp) pairs — no further de-duplication needed.

## 5. Aggregation

Raw data is one row per price *update*, so a station can have anywhere from a handful to hundreds of rows a day depending on how often its price changed. To build clean hourly/daily/monthly views, the data needs to be resampled onto a regular time grid.

### 5.1 Hourly averages

For each station, prices are floored to the hour and averaged within that hour. Stations only send an update *when the price changes*, so many station-hours have no update at all — for those, **forward-fill** carries the last known price forward, since a station with no update simply kept charging its previous price rather than having no price.

In [14]:
df_temp = df.copy()
df_temp['hour'] = df_temp['date'].dt.floor('h')

In [15]:
full_hours = pd.date_range(
    start='2025-12-01 00:00:00',
    end='2025-12-31 23:00:00',
    freq='h'
)

# Full grid for each station
stations = df['station_uuid'].unique()
full_index = pd.MultiIndex.from_product(
    [stations, full_hours],
    names=['station_uuid', 'hour']
)


In [16]:
hourly = (
    df_temp
    .groupby(['station_uuid', 'hour'], as_index=False)
    .agg({
        'diesel': 'mean',
        'e5': 'mean',
        'e10': 'mean'
    })
)

In [17]:
hourly_prices = (
    hourly
    .set_index(['station_uuid', 'hour'])
    .reindex(full_index)
    .reset_index()
)

In [18]:
hourly_prices[['diesel', 'e5', 'e10']] = (
    hourly_prices
    .groupby('station_uuid')[['diesel', 'e5', 'e10']]
    .ffill()
)

In [19]:
hourly_prices.head()

,station_uuid,hour,diesel,e5,e10
0,3116ea83-358d-4528-a440-6a84f56cde37,2025-12-01 00:00:00,1.619,1.759,1.699
1,3116ea83-358d-4528-a440-6a84f56cde37,2025-12-01 01:00:00,1.619,1.759,1.699
2,3116ea83-358d-4528-a440-6a84f56cde37,2025-12-01 02:00:00,1.619,1.759,1.699
3,3116ea83-358d-4528-a440-6a84f56cde37,2025-12-01 03:00:00,1.619,1.759,1.699
4,3116ea83-358d-4528-a440-6a84f56cde37,2025-12-01 04:00:00,1.619,1.759,1.699


### 5.2 Sanity check

After reindexing to a full station × hour grid and forward-filling, check how much is still missing. Any remaining gap should only be a station's *very first* hours in the month, before it has a price to carry forward.

In [20]:
hourly_prices.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11205384 entries, 0 to 11205383
Data columns (total 5 columns):
 #   Column        Dtype         
---  ------        -----         
 0   station_uuid  object        
 1   hour          datetime64[ns]
 2   diesel        float64       
 3   e5            float64       
 4   e10           float64       
dtypes: datetime64[ns](1), float64(3), object(1)
memory usage: 427.5+ MB


**Result:** 11,205,384 station-hour rows (744 hours × ~15,000 stations).

In [21]:
hourly_prices['diesel'].isna().sum()


186121

In [22]:
percentage = hourly_prices['diesel'].isna().sum() / 11205384 * 100
print(f"Percentage of empty values: {percentage:.2f}%")

Percentage of empty values: 1.66%


In [23]:
hourly_prices['e5'].isna().sum()

439734

In [24]:
percentage = hourly_prices['e5'].isna().sum() / 11205384 * 100
print(f"Percentage of empty values: {percentage:.2f}%")

Percentage of empty values: 3.92%


In [25]:
hourly_prices['e10'].isna().sum()

875044

In [26]:
percentage = hourly_prices['e10'].isna().sum() / 11205384 * 100
print(f"Percentage of empty values: {percentage:.2f}%")

Percentage of empty values: 7.81%


**Result:** remaining gaps are small — 1.66% (diesel), 3.92% (E5), 7.81% (E10) — consistent with the pattern already seen (E10 not sold everywhere), plus a handful of stations with no price yet at the very start of the month.

In [27]:
hourly_prices.describe()

,hour,diesel,e5,e10
count,11205384,1.101926e+07,1.076565e+07,1.033034e+07
mean,2025-12-16 11:29:59.999999744,1.594763e+00,1.707363e+00,1.650130e+00
min,2025-12-01 00:00:00,1.179000e+00,1.109000e+00,1.399000e+00
25%,2025-12-08 17:45:00,1.539000e+00,1.659000e+00,1.600667e+00
50%,2025-12-16 11:30:00,1.579000e+00,1.689000e+00,1.634000e+00
75%,2025-12-24 05:15:00,1.619000e+00,1.734000e+00,1.679000e+00
max,2025-12-31 23:00:00,3.330000e+00,3.330000e+00,3.330000e+00
std,NaN,9.733184e-02,9.292538e-02,9.307385e-02


### 5.3 Daily averages

Same logic as the hourly step, but floored to the day instead of the hour, to build the monthly trend chart. A `count` per fuel type is also kept alongside the mean, so days with very few price updates could be flagged if needed.

In [28]:
df_temp = df.copy()
df_temp['day'] = df_temp['date'].dt.floor('D')

full_days = pd.date_range(
    start='2025-12-01',
    end='2025-12-31',
    freq='D'
)


In [29]:
stations = df['station_uuid'].unique()
full_index = pd.MultiIndex.from_product(
    [stations, full_days],
    names=['station_uuid', 'day']
)

daily = (
    df_temp
    .groupby(['station_uuid', 'day'], as_index=False)
    .agg({
        'diesel': ['mean', 'count'],
        'e5': ['mean', 'count'],
        'e10': ['mean', 'count']
    })
)

daily.columns = ['station_uuid', 'day', 
                 'diesel', 'diesel_changes',
                 'e5', 'e5_changes', 
                 'e10', 'e10_changes']


In [30]:
daily_prices = (
    daily
    .set_index(['station_uuid', 'day'])
    .reindex(full_index)
    .reset_index()
)


daily_prices[['diesel', 'e5', 'e10']] = (
    daily_prices
    .groupby('station_uuid')[['diesel', 'e5', 'e10']]
    .ffill()
)

# For stations with no changes during day, to fill cells with 0.
daily_prices[['diesel_changes', 'e5_changes', 'e10_changes']] = (
    daily_prices[['diesel_changes', 'e5_changes', 'e10_changes']]
    .fillna(0)
)

In [31]:
daily_prices.head()

,station_uuid,day,diesel,diesel_changes,e5,e5_changes,e10,e10_changes
0,3116ea83-358d-4528-a440-6a84f56cde37,2025-12-01,1.636234,47.0,1.774957,47.0,1.714957,47.0
1,3116ea83-358d-4528-a440-6a84f56cde37,2025-12-02,1.606083,48.0,1.759417,48.0,1.699417,48.0
2,3116ea83-358d-4528-a440-6a84f56cde37,2025-12-03,1.589566,53.0,1.768057,53.0,1.708057,53.0
3,3116ea83-358d-4528-a440-6a84f56cde37,2025-12-04,1.609536,56.0,1.765429,56.0,1.705429,56.0
4,3116ea83-358d-4528-a440-6a84f56cde37,2025-12-05,1.598649,57.0,1.769877,57.0,1.709877,57.0


### 5.4 Sanity check

In [32]:
daily_prices.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 466891 entries, 0 to 466890
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   station_uuid    466891 non-null  object        
 1   day             466891 non-null  datetime64[ns]
 2   diesel          462569 non-null  float64       
 3   diesel_changes  466891 non-null  float64       
 4   e5              451917 non-null  float64       
 5   e5_changes      466891 non-null  float64       
 6   e10             433605 non-null  float64       
 7   e10_changes     466891 non-null  float64       
dtypes: datetime64[ns](1), float64(6), object(1)
memory usage: 28.5+ MB


**Result:** 466,891 station-day rows (31 days × ~15,000 stations).

In [33]:
daily_prices.describe()

,day,diesel,diesel_changes,e5,e5_changes,e10,e10_changes
count,466891,462569.000000,466891.000000,451917.000000,466891.000000,433605.000000,466891.000000
mean,2025-12-16 00:00:00,1.595529,29.224341,1.707998,28.762542,1.650724,27.990158
min,2025-12-01 00:00:00,1.408233,0.000000,1.239000,0.000000,1.399000,0.000000
25%,2025-12-08 00:00:00,1.556647,22.000000,1.671414,21.000000,1.614135,20.000000
50%,2025-12-16 00:00:00,1.582778,30.000000,1.694952,30.000000,1.637387,29.000000
75%,2025-12-24 00:00:00,1.612333,38.000000,1.720944,38.000000,1.663400,38.000000
max,2025-12-31 00:00:00,3.000000,165.000000,3.000000,165.000000,3.000000,165.000000
std,NaN,0.087955,15.184610,0.087524,15.617165,0.087723,16.140226


In [34]:
daily_prices['diesel'].isna().sum()


4322

In [35]:
percentage = daily_prices['diesel'].isna().sum() / 11205384 * 100
print(f"Percentage of empty values: {percentage:.2f}%")

Percentage of empty values: 0.04%


In [36]:
daily_prices['e5'].isna().sum()


14974

In [37]:
percentage = daily_prices['e5'].isna().sum() / 11205384 * 100
print(f"Percentage of empty values: {percentage:.2f}%")

Percentage of empty values: 0.13%


In [38]:
daily_prices['e10'].isna().sum()


33286

In [39]:
percentage = daily_prices['e10'].isna().sum() / 11205384 * 100
print(f"Percentage of empty values: {percentage:.2f}%")

Percentage of empty values: 0.30%


**Result:** at daily grain the missing share drops to near-zero (0.04–0.30%), since forward-fill has a full day to find a starting price for almost every station.

### 5.5 Full-month station averages

For the regional and brand comparisons, each station is collapsed to a single average price per fuel for the whole month — a simple mean over all valid raw updates. The hourly grid isn't needed here, since day-of-week and time-of-day effects average out over 31 days.

In [40]:
stations_avg = (
    df.groupby("station_uuid")[["diesel", "e5", "e10"]]
      .mean()
      .reset_index()
)

In [41]:
stations_avg.head()

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,station_uuid,diesel,e5,e10
0,00006210-0037-4444-8888-acdc00006210,1.614156,1.678973,1.618982
1,00016899-3247-4444-8888-acdc00000007,1.612372,1.709233,1.649814
2,00041414-208c-4444-8888-acdc00000414,1.553211,1.678474,NaN
3,00041450-0002-4444-8888-acdc00000002,1.548000,1.640222,1.600222
4,00047369-0001-4444-8888-acdc00000001,1.663050,1.724832,1.665426


This warning appears when pandas tries to format numbers for display and encounters invalid values such as **NaN**, **inf**, or **non‑numeric** data. It does not affect calculations — it only indicates that some values cannot be compared during formatting. No big deal for analysis

In [42]:
stations_avg.describe()

,diesel,e5,e10
count,15057.000000,14706.000000,14092.000000
mean,1.596629,1.708682,1.651412
std,0.085159,0.086324,0.086631
min,1.469000,1.239000,1.399000
25%,1.563131,1.675115,1.617501
50%,1.584197,1.695198,1.637603
75%,1.607333,1.717326,1.659954
max,3.000000,3.000000,3.000000


In [43]:
stations_avg['diesel'].isna().sum()

4

In [44]:
stations_avg['e5'].isna().sum()

355

In [45]:
stations_avg['e10'].isna().sum()

969

**Result:** 15,057 stations have a valid full-month diesel average; a smaller number lack a valid E5 (355) or E10 (969) average — again consistent with stations that simply don't sell that grade.

## 6. Export

Three cleaned/aggregated datasets are written out for use in the Tableau report.

In [46]:
hourly_prices.to_csv('tankkoenig_hourly_prices.csv', index=False)
daily_prices.to_csv('tankkoenig_daily_prices.csv', index=False)
stations_avg.to_csv('tankkoenig_station_times_prices.csv', index=False)

## Data Cleaning Summary

| Step | Result |
|---|---|
| Input | 31 daily CSVs → 13,648,295 raw price-update rows |
| Columns dropped | `dieselchange`, `e5change`, `e10change` (unused change-flags) |
| Timestamps | parsed & converted UTC → Europe/Berlin, timezone stripped; 0 rows failed to parse |
| Invalid prices removed | values ≤ 0€ → `NaN`: 3,713 diesel (0.03%), 219,323 E5 (1.6%), 579,942 E10 (4.2%) |
| Duplicates | 0 duplicate (station, timestamp) pairs |
| Outliers | **not removed** — high values (e.g. max E5 = 4.444€) are treated as genuine, since price differences of this size do occur at real stations (e.g. motorway locations); no evidence found that they're data errors |
| Hourly aggregation | 11,205,384 station-hour rows; 1.66–7.81% missing depending on fuel, mainly stations that don't sell that grade |
| Daily aggregation | 466,891 station-day rows; 0.04–0.30% missing |
| Station-level (full month) | 15,057 stations with a valid diesel average (14,706 E5, 14,092 E10) |
| Output | `tankkoenig_hourly_prices.csv`, `tankkoenig_daily_prices.csv`, `tankkoenig_station_times_prices.csv` |

**Known limitation:** missing E5/E10 values are assumed to mean *"this station doesn't sell this fuel type"* rather than a data error, based on the missing-rate pattern (diesel ≈0%, E10 up to ~4–8%). This wasn't independently verified against station metadata (e.g. checking whether the same `station_uuid` is *consistently* missing E10 across the whole month) — a reasonable next step if this pipeline were extended.